# 🔧 Heart Disease Dataset — Preprocessing

**Inputs:** `heart_disease.csv`  
**Outputs:** `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`  
**Steps:** Outlier handling → Feature engineering → Encoding → Scaling → Train-test split

---


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.dpi"] = 110

df = pd.read_csv("heart_disease.csv")
print(f"Original shape: {df.shape}")
df.head()


## 1. Verify No Missing Values

In [ ]:
print("Missing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

# Drop duplicates if any
df = df.drop_duplicates()
print(f"\nShape after dedup: {df.shape}")


## 2. Outlier Detection (IQR Method)

In [ ]:
continuous = ["age", "trestbps", "chol", "thalach", "oldpeak"]

fig, axes = plt.subplots(1, len(continuous), figsize=(16, 4))
outlier_summary = {}

for i, col in enumerate(continuous):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_summary[col] = {"lower": round(lower, 2), "upper": round(upper, 2), "n_outliers": n_out}
    axes[i].boxplot(df[col], patch_artist=True,
                    boxprops=dict(facecolor="#4C72B0", alpha=0.7))
    axes[i].set_title(f"{col}\n({n_out} outliers)")

plt.suptitle("Outlier Detection (IQR Fences)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nOutlier Summary:")
pd.DataFrame(outlier_summary).T


### Outlier Strategy
- `chol` has a few extreme values — **cap at IQR fences** (Winsorization) to preserve rows.
- `oldpeak` is right-skewed — apply **log1p** transformation after capping.
- Other features have minimal outliers; leave as-is.


In [ ]:
def winsorise(series):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    return series.clip(Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)

df_clean = df.copy()
for col in ["trestbps", "chol", "thalach"]:
    df_clean[col] = winsorise(df_clean[col])

print("Winsorisation applied to: trestbps, chol, thalach")
print(f"Shape unchanged: {df_clean.shape}")


## 3. Feature Engineering

In [ ]:
# ── New features ─────────────────────────────────────────────────────────
# Age buckets
df_clean["age_group"] = pd.cut(df_clean["age"],
                               bins=[0, 40, 50, 60, 100],
                               labels=["<40", "40-50", "50-60", "60+"])

# Log-transform right-skewed oldpeak
df_clean["oldpeak_log"] = np.log1p(df_clean["oldpeak"])

# High cholesterol flag
df_clean["high_chol"] = (df_clean["chol"] > 200).astype(int)

# Max HR ratio (observed / predicted max = 220 - age)
df_clean["hr_ratio"] = df_clean["thalach"] / (220 - df_clean["age"])

print("New features added: age_group, oldpeak_log, high_chol, hr_ratio")
df_clean[["age", "age_group", "oldpeak", "oldpeak_log", "chol", "high_chol", "thalach", "hr_ratio"]].head(5)


## 4. Encode Categorical Features

In [ ]:
# age_group → one-hot encode
df_encoded = pd.get_dummies(df_clean, columns=["age_group"], drop_first=False)

# Binary / ordinal cols are already numeric — nothing extra needed
print("Shape after encoding:", df_encoded.shape)
print("Columns:", df_encoded.columns.tolist())


## 5. Feature Selection — Remove Redundant

In [ ]:
# Drop original oldpeak (replaced by log version)
df_encoded = df_encoded.drop(columns=["oldpeak"])
print("Dropped: oldpeak (replaced by oldpeak_log)")
print("Final feature set shape:", df_encoded.drop(columns=["target"]).shape)


## 6. Train-Test Split

In [ ]:
X = df_encoded.drop(columns=["target"])
y = df_encoded["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}  y_test:  {y_test.shape}")
print(f"\nTrain class balance:\n{y_train.value_counts(normalize=True).round(3)}")
print(f"\nTest class balance:\n{y_test.value_counts(normalize=True).round(3)}")


## 7. Feature Scaling

In [ ]:
# Scale only continuous columns; binary/one-hot columns left unchanged
scale_cols = ["age", "trestbps", "chol", "thalach", "oldpeak_log", "hr_ratio"]
# Keep only scale_cols that actually exist in X after the pipeline
scale_cols = [c for c in scale_cols if c in X_train.columns]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()

X_train_scaled[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test_scaled[scale_cols]  = scaler.transform(X_test[scale_cols])

print("Scaled columns:", scale_cols)
X_train_scaled[scale_cols].describe().T[["mean","std"]].round(3)


## 8. Visualise Scaled Distributions

In [ ]:
fig, axes = plt.subplots(2, len(scale_cols), figsize=(18, 6))
for i, col in enumerate(scale_cols):
    axes[0, i].hist(X_train[col], bins=25, color="#4C72B0", edgecolor="white", alpha=0.8)
    axes[0, i].set_title(f"{col}\n(raw)", fontsize=9)
    axes[1, i].hist(X_train_scaled[col], bins=25, color="#DD8452", edgecolor="white", alpha=0.8)
    axes[1, i].set_title(f"{col}\n(scaled)", fontsize=9)
plt.suptitle("Before vs After Scaling", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


## 9. Save Processed Splits

In [ ]:
X_train_scaled.to_csv("X_train.csv", index=False)
X_test_scaled.to_csv("X_test.csv",  index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv",   index=False)

print("✅ Saved: X_train.csv, X_test.csv, y_train.csv, y_test.csv")
print(f"   X_train shape: {X_train_scaled.shape}")
print(f"   X_test  shape: {X_test_scaled.shape}")


## 10. Preprocessing Summary

| Step | Action |
|------|--------|
| Missing values | None — dataset is complete |
| Duplicates | Removed (if any) |
| Outliers | Winsorised `trestbps`, `chol`, `thalach` |
| Feature engineering | `age_group` bins, `oldpeak_log`, `high_chol`, `hr_ratio` |
| Encoding | One-hot on `age_group` |
| Scaling | StandardScaler on continuous features (fit on train only) |
| Split | 80% train / 20% test, stratified on target |

➡️ Proceed to `03_model_building.ipynb` for training and evaluation.
